# 📊 Tuần 8 — Failure Analysis & Metrics

**Project:** Dynamic Spatial Hand Gesture Recognition  
**Phase:** 6 / 6 — Post-Training Evaluation  
**Objective:** Deep-dive into model failures, per-class performance, and cross-model comparison.

---

## Notebook Outline

| # | Section | Purpose |
|---|---------|----------|
| 1 | Setup & Data Loading | Load test data, define model architectures, restore weights |
| 2 | Confusion Matrix | Count-based & normalized heatmaps |
| 3 | Misclassification Extractor | Identify & rank the most confidently wrong predictions |
| 4 | Classification Report | Precision / Recall / F1 per class — table & visual |
| 5 | Cross-Model Comparison | LSTM vs Transformer side-by-side analysis |
| 6 | Failure Pattern Analysis | In-depth discussion of systematic error sources |

---

## 1. Setup & Data Loading

In [ ]:
# ============================================================================
# 1.1  Imports
# ============================================================================
import os
import warnings
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {device}")
if device.type == "cuda":
    print(f"    GPU    : {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# 1.2  Paths
# ============================================================================
PROJECT_ROOT = r"D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training"
DATA_PATH    = os.path.join(PROJECT_ROOT, "Dataset", "raw_data.csv")
MODEL_DIR    = os.path.join(PROJECT_ROOT, "Models")

LSTM_WEIGHTS        = os.path.join(MODEL_DIR, "best_lstm.pth")
TRANSFORMER_WEIGHTS = os.path.join(MODEL_DIR, "best_transformer.pth")

print(f"📂 Dataset        : {DATA_PATH}")
print(f"📂 LSTM weights   : {LSTM_WEIGHTS}  (exists: {os.path.exists(LSTM_WEIGHTS)})")
print(f"📂 Transf. weights: {TRANSFORMER_WEIGHTS}  (exists: {os.path.exists(TRANSFORMER_WEIGHTS)})")

In [ ]:
# ============================================================================
# 1.3  Global Style Configuration
# ============================================================================
# Professional color palette used throughout the notebook
PALETTE = {
    "primary"   : "#2E86AB",  # Steel blue
    "secondary" : "#A23B72",  # Berry
    "accent"    : "#F18F01",  # Amber
    "success"   : "#2ECC71",  # Emerald
    "danger"    : "#E74C3C",  # Red
    "dark"      : "#2C3E50",  # Navy
    "light"     : "#ECF0F1",  # Cloud
}

# Matplotlib defaults
plt.rcParams.update({
    "figure.facecolor"   : "white",
    "axes.facecolor"     : "white",
    "axes.edgecolor"     : PALETTE["dark"],
    "axes.labelcolor"    : PALETTE["dark"],
    "xtick.color"        : PALETTE["dark"],
    "ytick.color"        : PALETTE["dark"],
    "font.family"        : "sans-serif",
    "font.size"          : 11,
    "axes.titlesize"     : 14,
    "axes.labelsize"     : 12,
    "figure.titlesize"   : 16,
    "figure.dpi"         : 120,
    "savefig.dpi"        : 150,
    "savefig.bbox"       : "tight",
})

# Seaborn theme
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)

print("✅ Visualization style configured.")

### 1.4 Load & Prepare Dataset

We load the full dataset and reproduce the **same** train / val / test split used during
training (80 / 10 / 10 with `random_state=42`) so that the test set is identical.

In [ ]:
# ============================================================================
# 1.4  Load CSV and reshape into (N_samples, 15, 63) tensors
# ============================================================================
raw_df = pd.read_csv(DATA_PATH)
print(f"Raw CSV shape : {raw_df.shape}")
print(f"Columns       : {list(raw_df.columns[:6])} ... {list(raw_df.columns[-3:])}")
print(f"Unique samples: {raw_df['sample_id'].nunique()}")
print(f"Unique labels : {sorted(raw_df['label'].unique())}")
raw_df.head()

In [ ]:
# ============================================================================
# 1.5  Pivot from long-form (N*15 rows) to sample-level arrays
# ============================================================================
SEQUENCE_LENGTH  = 15
NUM_LANDMARKS    = 21
NUM_COORDS       = 3
FEATURES_PER_FRAME = NUM_LANDMARKS * NUM_COORDS  # 63

# Landmark column names
landmark_cols = []
for i in range(NUM_LANDMARKS):
    landmark_cols.extend([f"x{i}", f"y{i}", f"z{i}"])

# Group by sample_id, ensuring frames are sorted by frame_index
grouped = raw_df.sort_values(["sample_id", "frame_index"]).groupby("sample_id")

samples = []   # Will hold (15, 63) arrays
labels  = []   # Corresponding label strings

for sid, group in grouped:
    if len(group) != SEQUENCE_LENGTH:
        continue  # Skip incomplete sequences
    seq = group[landmark_cols].values.astype(np.float32)   # (15, 63)
    samples.append(seq)
    labels.append(group["label"].iloc[0])

X_all = np.array(samples)    # (N, 15, 63)
y_raw = np.array(labels)     # (N,)

# Encode labels
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(y_raw)
class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print(f"\nDataset tensor : {X_all.shape}")
print(f"Classes ({num_classes}) : {class_names}")

In [ ]:
# ============================================================================
# 1.6  Reproduce the Train / Val / Test Split
# ============================================================================
# Same split ratios and random_state as in training notebooks
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.10, random_state=SEED, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1111,  # 0.1111 of 0.90 ≈ 0.10 overall
    random_state=SEED, stratify=y_train_val
)

print(f"Train : {X_train.shape[0]:>5d} samples")
print(f"Val   : {X_val.shape[0]:>5d} samples")
print(f"Test  : {X_test.shape[0]:>5d} samples")

# Convert to PyTorch tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
# ============================================================================
# 1.7  Simple PyTorch Dataset / DataLoader for batch inference
# ============================================================================
class GestureDataset(Dataset):
    """Wrap numpy arrays into a map-style PyTorch Dataset."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_dataset = GestureDataset(X_test, y_test)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"✅ Test DataLoader ready — {len(test_loader)} batches.")

### 1.8 Model Architecture Definitions

We re-define **both** architectures (LSTM and Transformer) here so that we can
load their saved `.pth` weights without importing from external modules.

In [ ]:
# ============================================================================
# 1.8a  LSTM Model Definition
# ============================================================================
class GestureLSTM(nn.Module):
    """
    Multi-layer LSTM for gesture sequence classification.
    Input  : (batch, seq_len=15, features=63)
    Output : (batch, num_classes)
    """
    def __init__(
        self,
        input_size: int = 63,
        hidden_size: int = 128,
        num_layers: int = 2,
        num_classes: int = 10,
        dropout: float = 0.3,
        bidirectional: bool = True,
    ):
        super().__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        # Layer normalisation on input features
        self.layer_norm = nn.LayerNorm(input_size)

        # LSTM backbone
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        # Attention-weighted pooling over time steps
        lstm_output_size = hidden_size * self.num_directions
        self.attention = nn.Sequential(
            nn.Linear(lstm_output_size, lstm_output_size // 2),
            nn.Tanh(),
            nn.Linear(lstm_output_size // 2, 1),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # x : (B, 15, 63)
        x = self.layer_norm(x)
        lstm_out, _ = self.lstm(x)  # (B, 15, hidden*dirs)

        # Attention pooling
        attn_weights = self.attention(lstm_out)        # (B, 15, 1)
        attn_weights = F.softmax(attn_weights, dim=1)  # normalise over time
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (B, hidden*dirs)

        logits = self.classifier(context)  # (B, num_classes)
        return logits

In [ ]:
# ============================================================================
# 1.8b  Transformer Model Definition
# ============================================================================
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for sequence data."""
    def __init__(self, d_model: int, max_len: int = 100, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class GestureTransformer(nn.Module):
    """
    Transformer encoder for gesture sequence classification.
    Input  : (batch, seq_len=15, features=63)
    Output : (batch, num_classes)
    """
    def __init__(
        self,
        input_size: int = 63,
        d_model: int = 128,
        nhead: int = 8,
        num_encoder_layers: int = 4,
        dim_feedforward: int = 256,
        num_classes: int = 10,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        # Linear projection from input features to d_model
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        # Learnable [CLS] token for classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x):
        # x : (B, 15, 63)
        B = x.size(0)
        x = self.input_projection(x)  # (B, 15, d_model)

        # Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, 16, d_model)

        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)  # (B, 16, d_model)

        cls_output = x[:, 0, :]  # Use CLS token output
        logits = self.classifier(cls_output)
        return logits

### 1.9 Load Saved Weights

We attempt to load both `.pth` files. If one is missing the notebook will
still run all single-model analyses on whichever model *is* available.

In [ ]:
# ============================================================================
# 1.9  Instantiate models & load weights
# ============================================================================
models = {}  # {name: model}

# ── LSTM ──────────────────────────────────────────────────────────────
if os.path.exists(LSTM_WEIGHTS):
    lstm_model = GestureLSTM(
        input_size=FEATURES_PER_FRAME,
        hidden_size=128,
        num_layers=2,
        num_classes=num_classes,
        dropout=0.3,
        bidirectional=True,
    ).to(device)
    lstm_state = torch.load(LSTM_WEIGHTS, map_location=device, weights_only=False)
    # Handle both raw state_dict and checkpoint-dict formats
    if isinstance(lstm_state, dict) and "model_state_dict" in lstm_state:
        lstm_model.load_state_dict(lstm_state["model_state_dict"])
    else:
        lstm_model.load_state_dict(lstm_state)
    lstm_model.eval()
    models["LSTM"] = lstm_model
    print("✅ LSTM model loaded successfully.")
else:
    print(f"⚠️  LSTM weights not found at {LSTM_WEIGHTS}")

# ── Transformer ──────────────────────────────────────────────────────
if os.path.exists(TRANSFORMER_WEIGHTS):
    tf_model = GestureTransformer(
        input_size=FEATURES_PER_FRAME,
        d_model=128,
        nhead=8,
        num_encoder_layers=4,
        dim_feedforward=256,
        num_classes=num_classes,
        dropout=0.1,
    ).to(device)
    tf_state = torch.load(TRANSFORMER_WEIGHTS, map_location=device, weights_only=False)
    if isinstance(tf_state, dict) and "model_state_dict" in tf_state:
        tf_model.load_state_dict(tf_state["model_state_dict"])
    else:
        tf_model.load_state_dict(tf_state)
    tf_model.eval()
    models["Transformer"] = tf_model
    print("✅ Transformer model loaded successfully.")
else:
    print(f"⚠️  Transformer weights not found at {TRANSFORMER_WEIGHTS}")

print(f"\n🔧 Models available for analysis: {list(models.keys())}")

if not models:
    raise FileNotFoundError(
        "No model weights found! Please train at least one model first "
        "(Tuan6_LSTM.ipynb or Tuan7_Transformer.ipynb)."
    )

In [ ]:
# ============================================================================
# 1.10  Inference Helper — returns labels, predictions, probabilities
# ============================================================================
@torch.no_grad()
def run_inference(model, dataloader):
    """
    Run a trained model over the full dataloader.

    Returns
    -------
    y_true  : np.ndarray (N,)   — ground-truth class indices
    y_pred  : np.ndarray (N,)   — predicted class indices
    y_probs : np.ndarray (N, C) — softmax probabilities
    """
    model.eval()
    all_true, all_pred, all_probs = [], [], []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        logits  = model(X_batch)                       # (B, C)
        probs   = F.softmax(logits, dim=1)             # (B, C)
        preds   = torch.argmax(probs, dim=1)           # (B,)

        all_true.append(y_batch.cpu().numpy())
        all_pred.append(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    return (
        np.concatenate(all_true),
        np.concatenate(all_pred),
        np.concatenate(all_probs),
    )


# Run inference for every available model
results = {}  # {model_name: (y_true, y_pred, y_probs)}

for name, model in models.items():
    y_true, y_pred, y_probs = run_inference(model, test_loader)
    acc = accuracy_score(y_true, y_pred)
    results[name] = {"y_true": y_true, "y_pred": y_pred, "y_probs": y_probs, "acc": acc}
    print(f"{name:12s} — Test Accuracy: {acc:.4f}  ({np.sum(y_true == y_pred)}/{len(y_true)})")

# Select primary model for detailed analysis (best accuracy)
primary_name = max(results, key=lambda k: results[k]["acc"])
primary = results[primary_name]
print(f"\n🏆 Primary model for detailed analysis: {primary_name}")

---

## 2. Confusion Matrix

A confusion matrix reveals which class pairs are most commonly confused.
We show **two views**:
1. **Count-based** — raw number of samples in each cell.
2. **Normalized** — each row sums to 100 %, making class-level recall directly readable.

In [ ]:
# ============================================================================
# 2.1  Compute Confusion Matrices
# ============================================================================
y_true = primary["y_true"]
y_pred = primary["y_pred"]

cm_counts = confusion_matrix(y_true, y_pred)
cm_norm   = confusion_matrix(y_true, y_pred, normalize="true") * 100  # percent

print(f"Confusion matrix shape: {cm_counts.shape}")

In [ ]:
# ============================================================================
# 2.2  Plot — Count-Based Confusion Matrix
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# ── Left: Raw Counts ─────────────────────────────────────────────────
ax1 = axes[0]
sns.heatmap(
    cm_counts,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Count"},
    ax=ax1,
)
ax1.set_title(f"Confusion Matrix — {primary_name} (Counts)", fontsize=15, fontweight="bold")
ax1.set_ylabel("True Label", fontsize=13)
ax1.set_xlabel("Predicted Label", fontsize=13)
ax1.tick_params(axis="x", rotation=45)
ax1.tick_params(axis="y", rotation=0)

# ── Right: Normalised (%) ────────────────────────────────────────────
ax2 = axes[1]
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Recall %"},
    ax=ax2,
)
ax2.set_title(f"Confusion Matrix — {primary_name} (Normalized %)", fontsize=15, fontweight="bold")
ax2.set_ylabel("True Label", fontsize=13)
ax2.set_xlabel("Predicted Label", fontsize=13)
ax2.tick_params(axis="x", rotation=45)
ax2.tick_params(axis="y", rotation=0)

fig.suptitle(
    f"Test-Set Confusion  |  Overall Accuracy = {primary['acc']:.2%}",
    fontsize=17, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

### Interpretation

* **Diagonal dominance** — Strong diagonal values indicate the model is generally
  performing well on most classes.
* **Off-diagonal clusters** — Any bright off-diagonal cells highlight systematic
  confusion between class pairs (e.g. `swipe_left` ↔ `swipe_right`).
* **Normalized view** — Eliminates class-imbalance bias; each row sums to 100 %,
  so you can directly read per-class recall.

---

## 3. Misclassification Extractor

We isolate every sample the model got wrong and rank them by **confidence** —
high-confidence errors are the most interesting because they suggest systematic
blind spots rather than noisy edge cases.

In [ ]:
# ============================================================================
# 3.1  Build Misclassification DataFrame
# ============================================================================
y_probs = primary["y_probs"]

# Confidence = probability assigned to the *predicted* class
pred_confidence = y_probs[np.arange(len(y_pred)), y_pred]

# Boolean mask for misclassified samples
wrong_mask = y_true != y_pred

misclass_df = pd.DataFrame({
    "sample_index"    : np.where(wrong_mask)[0],
    "true_label"      : label_encoder.inverse_transform(y_true[wrong_mask]),
    "predicted_label" : label_encoder.inverse_transform(y_pred[wrong_mask]),
    "confidence_score": pred_confidence[wrong_mask],
    "true_class_prob" : y_probs[wrong_mask, y_true[wrong_mask].astype(int)].diagonal()
                        if wrong_mask.sum() > 0 else [],
})

# Compute true_class_prob properly (prob the model assigned to the correct class)
if len(misclass_df) > 0:
    wrong_indices = np.where(wrong_mask)[0]
    true_probs = [y_probs[i, y_true[i]] for i in wrong_indices]
    misclass_df["true_class_prob"] = true_probs

misclass_df = misclass_df.sort_values("confidence_score", ascending=False).reset_index(drop=True)

total_wrong = len(misclass_df)
total_test  = len(y_true)
print(f"Misclassified : {total_wrong} / {total_test}  ({total_wrong/total_test:.2%})")
print(f"Correct       : {total_test - total_wrong} / {total_test}  ({(total_test - total_wrong)/total_test:.2%})")

In [ ]:
# ============================================================================
# 3.2  Top-20 Most Confidently Wrong Predictions
# ============================================================================
print("=" * 80)
print("  TOP-20 MOST CONFIDENTLY WRONG PREDICTIONS")
print("  (High confidence + wrong answer = systematic model blind spot)")
print("=" * 80)

top20 = misclass_df.head(20)
display(top20.style.format({
    "confidence_score": "{:.4f}",
    "true_class_prob" : "{:.4f}",
}).background_gradient(
    subset=["confidence_score"],
    cmap="Reds",
).set_caption("Top-20 Most Confidently Wrong Predictions"))

In [ ]:
# ============================================================================
# 3.3  Distribution of Misclassifications Per Class
# ============================================================================
if len(misclass_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))

    # ── Left: By True Label (which classes get misrecognised most?) ───
    true_counts = misclass_df["true_label"].value_counts().reindex(class_names, fill_value=0)
    colors_true = [PALETTE["danger"] if v == true_counts.max() else PALETTE["primary"]
                   for v in true_counts.values]

    ax1 = axes[0]
    bars1 = ax1.barh(true_counts.index, true_counts.values, color=colors_true, edgecolor="white")
    ax1.set_xlabel("Number of Misclassifications", fontsize=12)
    ax1.set_title("Misclassifications by True Label\n(Which gestures are hardest to recognise?)",
                  fontsize=13, fontweight="bold")
    ax1.invert_yaxis()
    for bar, val in zip(bars1, true_counts.values):
        if val > 0:
            ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     str(val), va="center", fontweight="bold", fontsize=10)

    # ── Right: By Predicted Label (what does the model falsely predict?) ──
    pred_counts = misclass_df["predicted_label"].value_counts().reindex(class_names, fill_value=0)
    colors_pred = [PALETTE["accent"] if v == pred_counts.max() else PALETTE["secondary"]
                   for v in pred_counts.values]

    ax2 = axes[1]
    bars2 = ax2.barh(pred_counts.index, pred_counts.values, color=colors_pred, edgecolor="white")
    ax2.set_xlabel("Number of False Predictions", fontsize=12)
    ax2.set_title("Misclassifications by Predicted Label\n(Which gestures are over-predicted?)",
                  fontsize=13, fontweight="bold")
    ax2.invert_yaxis()
    for bar, val in zip(bars2, pred_counts.values):
        if val > 0:
            ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     str(val), va="center", fontweight="bold", fontsize=10)

    plt.suptitle(f"Error Distribution — {primary_name} Model",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("🎉 No misclassifications! The model achieved 100% accuracy on the test set.")

In [ ]:
# ============================================================================
# 3.4  Confusion Pair Analysis — Most Common (True → Predicted) Error Pairs
# ============================================================================
if len(misclass_df) > 0:
    pair_counts = (
        misclass_df
        .groupby(["true_label", "predicted_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(15)
    )
    pair_counts["pair"] = pair_counts["true_label"] + " → " + pair_counts["predicted_label"]

    fig, ax = plt.subplots(figsize=(12, 6))
    colors_pair = sns.color_palette("coolwarm", n_colors=len(pair_counts))
    ax.barh(pair_counts["pair"][::-1], pair_counts["count"][::-1],
            color=colors_pair[::-1], edgecolor="white")
    ax.set_xlabel("Count", fontsize=12)
    ax.set_title("Top Confusion Pairs  (True Label → Predicted Label)",
                 fontsize=14, fontweight="bold")
    for i, (val, pair) in enumerate(zip(pair_counts["count"][::-1], pair_counts["pair"][::-1])):
        ax.text(val + 0.2, i, str(val), va="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

---

## 4. Classification Report

Sklearn's `classification_report` gives per-class **Precision**, **Recall**, and
**F1-Score** along with macro/weighted averages.

In [ ]:
# ============================================================================
# 4.1  Textual Classification Report
# ============================================================================
report_text = classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4,
)
print("=" * 72)
print(f"  CLASSIFICATION REPORT — {primary_name}")
print("=" * 72)
print(report_text)

In [ ]:
# ============================================================================
# 4.2  Per-Class Metrics — Visual Bar Charts
# ============================================================================
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=range(num_classes), zero_division=0
)

metrics_df = pd.DataFrame({
    "Class"     : class_names,
    "Precision" : precision,
    "Recall"    : recall,
    "F1-Score"  : f1,
    "Support"   : support.astype(int),
})

# Highlight worst-performing classes (lowest F1)
worst_f1_threshold = metrics_df["F1-Score"].quantile(0.25)
metrics_df["is_weak"] = metrics_df["F1-Score"] <= worst_f1_threshold

print("Per-class metrics:")
display(metrics_df.style.format({
    "Precision": "{:.4f}",
    "Recall"   : "{:.4f}",
    "F1-Score" : "{:.4f}",
}).apply(
    lambda row: ["background-color: #FADBD8" if row["is_weak"] else "" for _ in row],
    axis=1,
).set_caption("⚠️  Pink rows = bottom-25% F1-Score classes"))

In [ ]:
# ============================================================================
# 4.3  Grouped Bar Chart — Precision / Recall / F1 per class
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))

x = np.arange(num_classes)
bar_width = 0.25

bars_p = ax.bar(x - bar_width, precision, bar_width, label="Precision",
                color=PALETTE["primary"], edgecolor="white", alpha=0.9)
bars_r = ax.bar(x,              recall,    bar_width, label="Recall",
                color=PALETTE["accent"],  edgecolor="white", alpha=0.9)
bars_f = ax.bar(x + bar_width,  f1,        bar_width, label="F1-Score",
                color=PALETTE["success"], edgecolor="white", alpha=0.9)

# Highlight weak classes with red bottom border
for i in range(num_classes):
    if metrics_df.iloc[i]["is_weak"]:
        ax.axvspan(i - 0.4, i + 0.4, alpha=0.08, color="red")
        ax.text(i, max(precision[i], recall[i], f1[i]) + 0.03, "⚠️",
                ha="center", fontsize=14)

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=11)
ax.set_ylabel("Score", fontsize=13)
ax.set_ylim(0, 1.15)
ax.set_title(f"Per-Class Precision / Recall / F1-Score — {primary_name}",
             fontsize=15, fontweight="bold")
ax.legend(loc="upper right", fontsize=12, framealpha=0.9)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

# Add value labels on top of each bar
for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01,
                    f"{height:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 4.4  Radar (Spider) Chart — Per-Class F1 Score Profile
# ============================================================================
angles = np.linspace(0, 2 * np.pi, num_classes, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

f1_values = f1.tolist() + [f1[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
ax.fill(angles, f1_values, color=PALETTE["primary"], alpha=0.25)
ax.plot(angles, f1_values, color=PALETTE["primary"], linewidth=2, marker="o", markersize=6)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(class_names, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title(f"F1-Score Radar — {primary_name}", fontsize=14, fontweight="bold", pad=20)

# Draw concentric reference circles
for val in [0.25, 0.50, 0.75, 1.0]:
    ax.axhline(y=val, color="gray", alpha=0.2, linestyle="--")

plt.tight_layout()
plt.show()

---

## 5. Cross-Model Comparison (LSTM vs Transformer)

If both model weights are available, we compare their performance side-by-side
across multiple dimensions.

In [ ]:
# ============================================================================
# 5.1  Check Availability & Prepare Data
# ============================================================================
both_available = len(results) >= 2

if both_available:
    print("✅ Both LSTM and Transformer results are available — running comparison.")
    model_names_sorted = sorted(results.keys())  # consistent ordering
else:
    print(f"⚠️  Only {list(results.keys())} available. Cross-model comparison will be skipped.")
    print("    Train the other model to enable this section.")

In [ ]:
# ============================================================================
# 5.2  Side-by-Side Confusion Matrices
# ============================================================================
if both_available:
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))

    for idx, name in enumerate(model_names_sorted):
        r = results[name]
        cm = confusion_matrix(r["y_true"], r["y_pred"], normalize="true") * 100
        ax = axes[idx]
        cmap = "Blues" if idx == 0 else "Purples"
        sns.heatmap(
            cm, annot=True, fmt=".1f", cmap=cmap,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor="white",
            vmin=0, vmax=100,
            cbar_kws={"label": "Recall %"},
            ax=ax,
        )
        ax.set_title(f"{name}  (Acc = {r['acc']:.2%})", fontsize=14, fontweight="bold")
        ax.set_ylabel("True Label", fontsize=12)
        ax.set_xlabel("Predicted Label", fontsize=12)
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)

    fig.suptitle("Normalised Confusion Matrices — Side-by-Side Comparison",
                 fontsize=17, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# 5.3  Overall Accuracy Comparison Bar Chart
# ============================================================================
if both_available:
    fig, ax = plt.subplots(figsize=(8, 5))

    model_accs = {name: results[name]["acc"] for name in model_names_sorted}
    bar_colors = [PALETTE["primary"], PALETTE["secondary"]]

    bars = ax.bar(model_accs.keys(), model_accs.values(), color=bar_colors,
                  edgecolor="white", width=0.5)

    for bar, acc in zip(bars, model_accs.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{acc:.2%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Accuracy", fontsize=13)
    ax.set_title("Overall Test Accuracy — Model Comparison",
                 fontsize=15, fontweight="bold")
    ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Annotate the winner
    winner = max(model_accs, key=model_accs.get)
    diff = abs(list(model_accs.values())[0] - list(model_accs.values())[1])
    ax.text(0.5, 0.02, f"🏆 {winner} wins by {diff:.2%}",
            transform=ax.transAxes, ha="center", fontsize=12,
            style="italic", color=PALETTE["dark"])

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# 5.4  Per-Class Accuracy Comparison
# ============================================================================
if both_available:
    per_class_acc = {}
    for name in model_names_sorted:
        r = results[name]
        cm = confusion_matrix(r["y_true"], r["y_pred"])
        # Per-class accuracy = diagonal / row sum
        row_sums = cm.sum(axis=1)
        # Avoid division by zero for classes with no test samples
        class_acc = np.where(row_sums > 0, cm.diagonal() / row_sums, 0.0)
        per_class_acc[name] = class_acc

    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(num_classes)
    bar_w = 0.35

    for i, name in enumerate(model_names_sorted):
        offset = (i - 0.5) * bar_w
        color  = bar_colors[i]
        bars = ax.bar(x + offset, per_class_acc[name], bar_w,
                      label=name, color=color, edgecolor="white", alpha=0.9)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                    f"{h:.0%}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=11)
    ax.set_ylabel("Accuracy", fontsize=13)
    ax.set_ylim(0, 1.15)
    ax.set_title("Per-Class Accuracy — LSTM vs Transformer", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12, framealpha=0.9)
    ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ── Identify classes where models disagree most ──────────────────
    diff = per_class_acc[model_names_sorted[0]] - per_class_acc[model_names_sorted[1]]
    diff_df = pd.DataFrame({
        "Class": class_names,
        model_names_sorted[0]: per_class_acc[model_names_sorted[0]],
        model_names_sorted[1]: per_class_acc[model_names_sorted[1]],
        "Difference": diff,
        "abs_diff": np.abs(diff),
    }).sort_values("abs_diff", ascending=False)

    print("\n📊 Classes with largest accuracy gap between models:")
    display(diff_df.drop(columns="abs_diff").style.format({
        model_names_sorted[0]: "{:.2%}",
        model_names_sorted[1]: "{:.2%}",
        "Difference": "{:+.2%}",
    }).background_gradient(subset=["Difference"], cmap="RdYlGn", vmin=-0.3, vmax=0.3))

In [ ]:
# ============================================================================
# 5.5  Per-Class F1 Comparison Radar (Both Models Overlaid)
# ============================================================================
if both_available:
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"polar": True})

    angles = np.linspace(0, 2 * np.pi, num_classes, endpoint=False).tolist()
    angles += angles[:1]

    colors_radar = [PALETTE["primary"], PALETTE["secondary"]]

    for i, name in enumerate(model_names_sorted):
        r = results[name]
        _, _, f1_vals, _ = precision_recall_fscore_support(
            r["y_true"], r["y_pred"], labels=range(num_classes), zero_division=0
        )
        values = f1_vals.tolist() + [f1_vals[0]]
        ax.fill(angles, values, color=colors_radar[i], alpha=0.15)
        ax.plot(angles, values, color=colors_radar[i], linewidth=2,
                marker="o", markersize=5, label=f"{name} (F1)")

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(class_names, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_title("F1-Score Radar — Both Models", fontsize=14, fontweight="bold", pad=25)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.05), fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# 5.6  Agreement Analysis — Where Do Both Models Fail Together?
# ============================================================================
if both_available:
    names = model_names_sorted
    r0, r1 = results[names[0]], results[names[1]]

    correct_0 = (r0["y_true"] == r0["y_pred"])
    correct_1 = (r1["y_true"] == r1["y_pred"])

    both_correct = np.sum(correct_0 & correct_1)
    both_wrong   = np.sum(~correct_0 & ~correct_1)
    only_0_right = np.sum(correct_0 & ~correct_1)
    only_1_right = np.sum(~correct_0 & correct_1)
    total = len(r0["y_true"])

    agreement_data = {
        "Both Correct"          : both_correct,
        "Both Wrong"            : both_wrong,
        f"Only {names[0]} Correct": only_0_right,
        f"Only {names[1]} Correct": only_1_right,
    }

    fig, ax = plt.subplots(figsize=(8, 8))
    wedge_colors = [PALETTE["success"], PALETTE["danger"], PALETTE["primary"], PALETTE["secondary"]]
    wedges, texts, autotexts = ax.pie(
        agreement_data.values(),
        labels=agreement_data.keys(),
        colors=wedge_colors,
        autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * total / 100))})",
        startangle=90,
        textprops={"fontsize": 11},
        pctdistance=0.75,
        wedgeprops={"edgecolor": "white", "linewidth": 2},
    )
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight("bold")

    ax.set_title("Model Agreement Analysis", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # If there are samples both models get wrong, those are truly hard
    if both_wrong > 0:
        both_wrong_mask = ~correct_0 & ~correct_1
        hard_samples = pd.DataFrame({
            "sample_index"            : np.where(both_wrong_mask)[0],
            "true_label"              : label_encoder.inverse_transform(r0["y_true"][both_wrong_mask]),
            f"{names[0]}_prediction"  : label_encoder.inverse_transform(r0["y_pred"][both_wrong_mask]),
            f"{names[1]}_prediction"  : label_encoder.inverse_transform(r1["y_pred"][both_wrong_mask]),
            f"{names[0]}_confidence"  : r0["y_probs"][both_wrong_mask].max(axis=1),
            f"{names[1]}_confidence"  : r1["y_probs"][both_wrong_mask].max(axis=1),
        })
        print(f"\n🔴 {both_wrong} samples misclassified by BOTH models:")
        display(hard_samples.head(20))

---

## 6. Failure Pattern Analysis — Detailed Commentary

This section provides an in-depth technical analysis of common failure modes
in 3D hand gesture time-series classification. Understanding *why* errors occur
is just as important as measuring *how many* errors occur.

---

### 6.1 Similar Spatial Trajectories Between Gesture Classes

**Problem:** Several gesture classes share nearly identical hand trajectories in
3D space and differ only in subtle wrist orientation or finger articulation.

| Confused Pair | Shared Trait | Differentiating Signal |
|---|---|---|
| `swipe_left` ↔ `swipe_right` | Horizontal wrist translation | Direction of movement (sign of Δx) |
| `swipe_up` ↔ `swipe_down` | Vertical wrist translation | Direction of movement (sign of Δy) |
| `rotate_cw` ↔ `rotate_ccw` | Circular wrist trajectory | Chirality (clockwise vs counter-clockwise) |
| `grab` ↔ `pinch` | Fingers closing inward | Number of fingers involved, thumb-index distance |

When the model collapses these differences (e.g. due to insufficient feature
discrimination in the embedding space), mirror-gesture pairs dominate the
off-diagonal entries of the confusion matrix.

**Recommendation:**
- Compute **first-order differences** (velocity features) alongside raw coordinates
  so that direction of motion is explicitly encoded.
- Add **relative landmark distances** (e.g. thumb–index distance) as auxiliary features.

---

### 6.2 Temporal Aliasing — Speed Variation

**Problem:** The same gesture performed slowly vs quickly occupies different
proportions of the fixed 15-frame window:

- A **fast** gesture may complete by frame 5, leaving frames 6-14 as idle/static.
- A **slow** gesture may still be in progress at frame 14, getting truncated.

Since both LSTM and Transformer receive fixed-length sequences, they have no
built-in mechanism to normalise for execution speed. The model effectively learns
a *speed-dependent* representation rather than a *gesture-dependent* one.

**Recommendation:**
- Apply **temporal interpolation** during preprocessing to re-sample all
  sequences to a canonical speed (e.g. Dynamic Time Warping alignment).
- Use **data augmentation** with random temporal scaling (speed 0.7× – 1.3×).

---

### 6.3 Camera Angle Sensitivity for 3D Coordinates

**Problem:** MediaPipe returns normalised coordinates relative to the **camera frame**,
not the user's body frame. Identical gestures viewed from different angles produce
different coordinate patterns:

- Rotating the wrist 30° toward the camera dramatically changes the Z-coordinate
  profile of all 21 landmarks.
- Users sitting at different distances change the scale of X/Y coordinates.

**Recommendation:**
- Normalise each frame to a **wrist-centred coordinate system** (subtract the
  wrist landmark from all 21 points).
- Apply **scale normalisation** (divide by palm width or hand bounding-box size).
- Augment training data with random 3D rotations and translations.

---

### 6.4 Boundary Effects — Gesture Start/End Ambiguity

**Problem:** The 15-frame recording window assumes a clean gesture onset at
frame 0 and completion at frame 14. In practice:

- The user may press SPACE slightly **before** or **after** the gesture begins.
- Transition frames (moving hand into position) may contaminate the sequence.
- The gesture may not reach its natural endpoint within 15 frames.

These boundary artefacts inject label noise that disproportionately affects
classes with slower or more complex motion patterns.

**Recommendation:**
- Implement **gesture spotting** (automatic onset detection) rather than
  manual trigger.
- Add temporal padding and use a **sliding-window classifier** that scores
  multiple offsets and picks the best alignment.

---

### 6.5 Dynamic Time Warping (DTW) Considerations

**Problem:** DTW is a classic technique for time-series comparison that can handle
speed variation and phase shifts. However:

- Neural networks (LSTM/Transformer) do **not** perform DTW internally — they
  use fixed positional indices.
- Gestures that are DTW-similar but frame-index-dissimilar will confuse the model.

**Recommendation:**
- Use DTW as a **pre-processing alignment step** to warp all sequences to a
  canonical template before feeding into the neural network.
- Alternatively, train a **DTW-inspired attention** mechanism that learns
  non-linear time warping end-to-end.

---

### 6.6 Data Quality & Collection Artefacts

**Problem:** The webcam-based collection pipeline introduces several noise sources:

| Source | Effect |
|---|---|
| MediaPipe hand detection failure | Zero-padded frames (all landmarks = 0.0) |
| Lighting changes | Jittery landmark positions between consecutive frames |
| Single-user bias | Model overfits to one person's hand size / skin tone |
| Label noise | User mislabels a gesture or performs it inconsistently |

**Recommendation:**
- Filter out sequences containing more than 2 zero-padded frames.
- Collect data from **multiple users** with diverse hand sizes.
- Apply **Gaussian noise augmentation** (σ = 0.01–0.03) to coordinates during training.

---

### 6.7 Improvement Roadmap

Based on the failure analysis above, the following priority actions are recommended:

| Priority | Action | Expected Impact |
|:---:|---|---|
| 🔴 High | Add velocity features (Δx, Δy, Δz per frame) | Resolves mirror-gesture confusion |
| 🔴 High | Wrist-centred + scale normalisation | Camera-invariant representations |
| 🟠 Medium | Temporal augmentation (random speed ±30%) | Speed-invariant classification |
| 🟠 Medium | Gaussian coordinate noise augmentation | Regularisation against jitter |
| 🟡 Low | Ensemble (LSTM + Transformer majority vote) | +1–3% accuracy from complementary strengths |
| 🟡 Low | Collect multi-user data (≥5 subjects) | Reduces single-user bias |
| 🟢 Future | DTW-based pre-alignment | Phase-invariant sequences |
| 🟢 Future | Graph Neural Network on hand skeleton | Exploits anatomical adjacency |

---

## 7. Summary & Conclusion

In [ ]:
# ============================================================================
# 7.1  Final Summary Table
# ============================================================================
print("=" * 72)
print("  FINAL SUMMARY — FAILURE ANALYSIS (WEEK 8)")
print("=" * 72)

for name in sorted(results.keys()):
    r = results[name]
    n_correct = np.sum(r["y_true"] == r["y_pred"])
    n_total   = len(r["y_true"])
    n_wrong   = n_total - n_correct

    _, _, f1_vals, _ = precision_recall_fscore_support(
        r["y_true"], r["y_pred"], labels=range(num_classes), zero_division=0
    )
    worst_class = class_names[np.argmin(f1_vals)]
    best_class  = class_names[np.argmax(f1_vals)]

    print(f"\n  📌 {name}")
    print(f"     Accuracy       : {r['acc']:.4f}  ({n_correct}/{n_total})")
    print(f"     Errors         : {n_wrong}")
    print(f"     Macro F1       : {np.mean(f1_vals):.4f}")
    print(f"     Best class     : {best_class}  (F1 = {np.max(f1_vals):.4f})")
    print(f"     Worst class    : {worst_class}  (F1 = {np.min(f1_vals):.4f})")

print("\n" + "=" * 72)
print("  Analysis complete. See Section 6 for detailed failure commentary.")
print("=" * 72)

In [ ]:
# ============================================================================
# 7.2  Confidence Distribution — Correct vs Incorrect Predictions
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 6))

y_true_p = primary["y_true"]
y_pred_p = primary["y_pred"]
max_probs = primary["y_probs"].max(axis=1)

correct_mask_p = y_true_p == y_pred_p

ax.hist(max_probs[correct_mask_p], bins=50, alpha=0.7, label="Correct",
        color=PALETTE["success"], edgecolor="white")
ax.hist(max_probs[~correct_mask_p], bins=50, alpha=0.7, label="Incorrect",
        color=PALETTE["danger"], edgecolor="white")

ax.set_xlabel("Max Predicted Probability (Confidence)", fontsize=13)
ax.set_ylabel("Count", fontsize=13)
ax.set_title(f"Confidence Distribution — {primary_name}",
             fontsize=15, fontweight="bold")
ax.legend(fontsize=12)
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="50% threshold")

# Annotate
mean_correct = max_probs[correct_mask_p].mean()
mean_wrong   = max_probs[~correct_mask_p].mean() if (~correct_mask_p).sum() > 0 else 0
ax.text(0.98, 0.95,
        f"Avg confidence (correct):   {mean_correct:.3f}\n"
        f"Avg confidence (incorrect): {mean_wrong:.3f}",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=11, bbox={"facecolor": "white", "alpha": 0.8, "boxstyle": "round"})

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 7.3  Per-Class Confidence Box Plot
# ============================================================================
fig, ax = plt.subplots(figsize=(14, 6))

conf_data = []
for cls_idx in range(num_classes):
    mask = y_true_p == cls_idx
    if mask.sum() > 0:
        conf_data.append(max_probs[mask])
    else:
        conf_data.append(np.array([0]))

bp = ax.boxplot(conf_data, labels=class_names, patch_artist=True,
                medianprops={"color": "black", "linewidth": 2},
                whiskerprops={"linewidth": 1.5},
                flierprops={"marker": "o", "markersize": 4, "alpha": 0.5})

# Color boxes by median confidence
medians = [np.median(d) for d in conf_data]
norm = plt.Normalize(vmin=min(medians), vmax=max(medians))
cmap = plt.cm.RdYlGn
for patch, med in zip(bp["boxes"], medians):
    patch.set_facecolor(cmap(norm(med)))
    patch.set_alpha(0.8)

ax.set_ylabel("Max Predicted Probability", fontsize=13)
ax.set_title(f"Per-Class Confidence Distribution — {primary_name}",
             fontsize=15, fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

print("\n✅ Failure analysis notebook complete.")